# PTV stops: load and clean

Flattens `public_transport_stops.geojson` into a stop-level table,
restricts to train and bus stops, removes non-platform station
amenities (car parks, bike racks, decision points, rail-replacement
bus stops), drops bad coordinates, deduplicates, and exports in the
`Facility ID`/`Latitude`/`Longitude` shape used by every other
destination file in the proximity pipeline.

## Imports and configuration

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.neighbors import BallTree

SRC_PATH = Path("../external_data/public_transport_stops.geojson")
OUT_PATH = Path("../external_data/ptv_stops.csv")

KEPT_MODES = {
    "METRO BUS",
    "REGIONAL BUS",
    "METRO TRAIN",
    "REGIONAL TRAIN",
}

VIC_LAT_RANGE = (-39.3, -33.9)
VIC_LON_RANGE = (140.9, 150.1)
EARTH_RADIUS_M = 6_371_000
DEDUP_DISTANCE_M = 50

## Read the GeoJSON

The source only carries three properties per feature: `STOP_ID`,
`STOP_NAME`, `MODE`. There is no field distinguishing a boarding
platform from a station amenity, so that distinction has to be made
from the name later.

In [2]:
with open(SRC_PATH) as geojson_file:
    geojson = json.load(geojson_file)

features = geojson["features"]
print(f"Raw features: {len(features)}")
print(features[0]["properties"].keys())

Raw features: 32035
dict_keys(['STOP_ID', 'STOP_NAME', 'MODE'])


## Flatten to a table

`STOP_ID` is documented as an integer, but train stops use GTFS-style
string IDs like `"vic:rail:ART"`, so it's kept as a string throughout.

In [3]:
def feature_to_row(feature):
    """Flatten one GeoJSON feature into a stop_id, name, mode and
    location.
    """
    properties = feature.get("properties", {})
    coordinates = feature.get("geometry", {}).get("coordinates")
    return {
        "stop_id": properties.get("STOP_ID"),
        "stop_name": properties.get("STOP_NAME"),
        "mode": properties.get("MODE"),
        "stop_lon": coordinates[0] if coordinates else None,
        "stop_lat": coordinates[1] if coordinates else None,
    }


stops_raw = pd.DataFrame(feature_to_row(f) for f in features)

In [4]:
stops_raw["stop_id"] = stops_raw["stop_id"].astype("string").str.strip()
stops_raw["stop_name"] = stops_raw["stop_name"].astype("string").str.strip()
stops_raw["mode"] = stops_raw["mode"].astype("string").str.strip()
stops_raw["stop_lat"] = pd.to_numeric(stops_raw["stop_lat"], errors="coerce")
stops_raw["stop_lon"] = pd.to_numeric(stops_raw["stop_lon"], errors="coerce")

## Drop rows with missing IDs or coordinates

In [5]:
def find_missing_rows(stops):
    """Return a mask of rows missing a stop_id or a coordinate."""
    missing_id = stops["stop_id"].isna() | (stops["stop_id"] == "")
    missing_coordinates = stops["stop_lat"].isna() | stops["stop_lon"].isna()
    return missing_id | missing_coordinates


is_missing = find_missing_rows(stops_raw)
print(f"Dropping {is_missing.sum()} rows missing stop_id or coordinates")

stops = stops_raw[~is_missing].reset_index(drop=True)

Dropping 0 rows missing stop_id or coordinates


## Look at rows outside Victoria before dropping

List them first, in case any are obvious errors (a sign flip, a
swapped lat/lon, a misplaced decimal) worth fixing by hand rather than
losing the stop entirely.

In [6]:
def find_outside_victoria(stops):
    """Return a mask of rows outside Victoria's bounding box."""
    is_in_victoria = (
        stops["stop_lat"].between(*VIC_LAT_RANGE)
        & stops["stop_lon"].between(*VIC_LON_RANGE)
    )
    return ~is_in_victoria


is_outside_victoria = find_outside_victoria(stops)
print(f"{is_outside_victoria.sum()} rows outside Victoria")

stops.loc[
    is_outside_victoria,
    ["stop_id", "stop_name", "mode", "stop_lat", "stop_lon"],
]

32 rows outside Victoria


,stop_id,stop_name,mode,stop_lat,stop_lon
630,20600,Adelaide Station,INTERSTATE TRAIN,-34.938167,138.581311
631,20609,Murray Bridge Station,INTERSTATE TRAIN,-35.116644,139.275554
632,20614,Bordertown Station,INTERSTATE TRAIN,-36.310557,140.777622
654,sa:rail:ADE,Adelaide Railway Station,INTERSTATE TRAIN,-34.938174,138.581288
655,sa:rail:BOR,Bordertown Railway Station,INTERSTATE TRAIN,-36.308607,140.773885
656,sa:rail:MUR,Murray Bridge Railway Station,INTERSTATE TRAIN,-35.116941,139.275285
28149,17749,Tourist Information Centre/Jubilee Hwy East,REGIONAL COACH,-37.825718,140.787002
28282,20807,Sweetie's Bakery/North Tce,REGIONAL COACH,-36.308471,140.772996
28283,20808,Makin St/Heritage St,REGIONAL COACH,-36.098623,140.355162
28284,20809,Tintinara Travel Stop,REGIONAL COACH,-35.883983,140.055779


In [7]:
stops = stops[~is_outside_victoria].reset_index(drop=True)

## Restrict to train and bus stops only

Tram, coach, skybus and interstate train are dropped, keeping only
metro/regional bus and metro/regional train.

In [8]:
is_kept_mode = stops["mode"].isin(KEPT_MODES)
print(f"Dropping {(~is_kept_mode).sum()} rows outside {KEPT_MODES}")

stops = stops[is_kept_mode].reset_index(drop=True)
print(stops["mode"].value_counts())

Dropping 2572 rows outside {'METRO BUS', 'REGIONAL TRAIN', 'METRO TRAIN', 'REGIONAL BUS'}
mode
METRO BUS         22617
REGIONAL BUS       3327
METRO TRAIN        2860
REGIONAL TRAIN      627
Name: count, dtype: Int64


## Require "Station" in the name for train stops

Train mode rows without "Station" in the name are near-universally
non-platform: street-level access points, decision points (including
abbreviated "DP" forms), lifts, and other station-precinct
infrastructure. Rather than continue to enumerate every junk pattern,
train rows are required to contain "Station" in the name.

In [9]:
is_train = stops["mode"].isin({"METRO TRAIN", "REGIONAL TRAIN"})
has_station_in_name = stops["stop_name"].str.contains(
    "Station", na=False, case=False
)

is_junk_train_row = is_train & ~has_station_in_name
print(f"Dropping {is_junk_train_row.sum()} non-platform train rows")

stops = stops[~is_junk_train_row].reset_index(drop=True)

Dropping 2351 non-platform train rows


In [10]:
STATION_STREET_NAMES = ("Station St", "Station Pl")

is_train = stops["mode"].isin({"METRO TRAIN", "REGIONAL TRAIN"})
is_station_street_name = stops["stop_name"].str.fullmatch(
    "|".join(STATION_STREET_NAMES), case=False, na=False
)
is_station_street = is_train & is_station_street_name

print(f"Dropping {is_station_street.sum()} street-named train rows")

stops = stops[~is_station_street].reset_index(drop=True)

Dropping 46 street-named train rows


In [11]:
is_train = stops["mode"].isin({"METRO TRAIN", "REGIONAL TRAIN"})
is_rail_replacement = is_train & stops["stop_name"].str.contains(
    "Rail Replacement", case=False, na=False
)
print(f"Dropping {is_rail_replacement.sum()} rail replacement train rows")

stops = stops[~is_rail_replacement].reset_index(drop=True)

Dropping 3 rail replacement train rows


## Deduplicate by ID, then by location

A duplicate `stop_id` is dropped outright. Separately, two different
IDs can represent the same physical stop (for example a stop mapped
once per direction of travel), so duplicates are also found by
proximity: any pair within `DEDUP_DISTANCE_M` of each other is treated
as the same stop, and only the first of each group is kept.

In [12]:
n_duplicate_ids = stops["stop_id"].duplicated().sum()
print(f"Dropping {n_duplicate_ids} duplicate stop_id rows")

stops = stops.drop_duplicates(subset="stop_id", keep="first")
stops = stops.reset_index(drop=True)

Dropping 226 duplicate stop_id rows


In [13]:
def deduplicate_by_location(stops, distance_m=DEDUP_DISTANCE_M):
    """Drop rows whose coordinates fall within distance_m of an
    earlier row, treating them as the same physical stop.
    """
    if len(stops) < 2:
        return stops

    coordinates_rad = np.radians(stops[["stop_lat", "stop_lon"]].to_numpy())
    tree = BallTree(coordinates_rad, metric="haversine")
    radius_rad = distance_m / EARTH_RADIUS_M

    is_duplicate = np.zeros(len(stops), dtype=bool)
    for row in range(len(stops)):
        if is_duplicate[row]:
            continue
        neighbours = tree.query_radius(
            coordinates_rad[[row]], r=radius_rad
        )[0]
        later_neighbours = neighbours[neighbours > row]
        is_duplicate[later_neighbours] = True

    print(f"Dropping {is_duplicate.sum()} of {len(stops)} as duplicates")
    return stops[~is_duplicate].reset_index(drop=True)


stops = deduplicate_by_location(stops)

Dropping 8515 of 26805 as duplicates


## Standardise columns and export

Renamed to `Facility ID`, `Latitude` and `Longitude` to match every
other destination file the proximity pipeline reads.

In [14]:
STANDARD_COLUMNS = {
    "stop_id": "Facility ID",
    "stop_name": "name",
    "mode": "mode",
    "stop_lat": "Latitude",
    "stop_lon": "Longitude",
}

stops_export = stops.rename(columns=STANDARD_COLUMNS)
stops_export = stops_export[list(STANDARD_COLUMNS.values())]

print(f"Facility ID unique: {stops_export['Facility ID'].is_unique}")
print(stops_export.isna().sum())

Facility ID unique: True
Facility ID    0
name           0
mode           0
Latitude       0
Longitude      0
dtype: int64


In [15]:
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
stops_export.to_csv(OUT_PATH, index=False)

print(f"\nFinal row count: {len(stops_export)}")
print("\nMode value counts:")
print(stops_export["mode"].value_counts(dropna=False).to_string())
print(f"\nWrote {OUT_PATH}")


Final row count: 18290

Mode value counts:
mode
METRO BUS         15322
REGIONAL BUS       2593
METRO TRAIN         236
REGIONAL TRAIN      139

Wrote ../external_data/ptv_stops.csv
